# Conversation validation examples

This notebook covers the complete checker usage surface: reports, builtin rule families, severity overrides, rule discovery, custom checkers, and report serialization. Launch it from the repository root with `uv run --with jupyterlab jupyter lab examples/validation_examples.ipynb`.

The library validates conversations on **two levels**:

1. **Message level** (eager, raising): pydantic validation at
   construction. Field grammar, id shapes, payload types, header
   rules. An object that violates these *cannot exist*; you get a
   `ValidationError` immediately.
2. **Cross-message level** (reported, never raised): checker rules
   span messages or need declared context, such as
   canonical input order, tool-result echo discipline, or tool-call
   args against the declared tool schema. These live in `check()`,
   which returns a `ConformanceReport` of violations. Non-conformant
   conversations stay representable by design: a checker can only
   describe what it can hold.

Each cross-message rule family is one checker class under
`apertus_common.checker.rules`, with a stable rule id and a default
severity (`error` breaks conformance, `warning` does not).

In [1]:
import json
from collections.abc import Iterable
from dataclasses import asdict
from typing import ClassVar

from pydantic import ValidationError

from apertus_common import (
    Assistant,
    Checker,
    ConformanceReport,
    Conversation,
    Effort,
    Harness,
    IdConventionChecker,
    InputOrderChecker,
    Location,
    Rule,
    Severity,
    SystemPrompt,
    TextPart,
    ToolCall,
    ToolResult,
    ToolResultChecker,
    ToolSpec,
    User,
    VerifiableAnswer,
    Violation,
    Wait,
    check,
    default_checkers,
    rule_table,
)

## 1. Message-level validation (raises at construction)

Bad field content never produces an object. A header value with
whitespace violates the header grammar:

In [2]:
try:
    Harness(payload="x", extras={"lang": "two words"})
except ValidationError as error:
    print(error)

1 validation error for Harness
extras.lang
  Value error, header value 'two words' contains whitespace [type=value_error, input_value='two words', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


A tool-call id outside the permissive shape is rejected the same way:

In [3]:
try:
    ToolCall(id="bad id!", name="bash", args={})
except ValidationError as error:
    print(error)

1 validation error for ToolCall
id
  Value error, tool-call id 'bad id!' does not match the permissive id shape [A-Za-z0-9_:.-]{1,64} [type=value_error, input_value='bad id!', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error


## 2. Cross-message rules (reported by `check()`)

These conversations construct fine; the rules they break span
messages. `check()` runs a strict default set of checkers.

**Canonical input order**: inputs at one boundary
arrive as harness, tool_result, retrieval/event, attachment, user.

In [4]:
messy = Conversation(
    items=[User(payload=[TextPart(text="hi")]), Harness(payload="notice")]
)
report = check(messy)
for violation in report:
    print(f"{violation.severity.upper():7} {violation.rule:35} at {violation.location}")
print("conformant:", report.ok)

ERROR   input-order/canonical-order         at items[1]
conformant: False


The correcting counterpart is explicit, never automatic:
`normalized()` fixes the order, and the same conversation passes.

In [5]:
print("after normalized():", check(messy.normalized()).ok)

after normalized(): True


**Tool calls against the declared inventory**: the
called name must be declared, and args must conform to that tool's
JSON Schema:

In [6]:
system = SystemPrompt(
    effort=Effort.MEDIUM,
    tools=[
        ToolSpec(
            name="get_weather",
            json_schema={
                "type": "object",
                "properties": {"city": {"type": "string"}},
                "required": ["city"],
                "additionalProperties": False,
            },
        )
    ],
)
bad_args = Conversation(
    system=system,
    items=[
        User(payload=[TextPart(text="weather?")]),
        ToolCall(id="get_weather:0", name="get_weather", args={"city": 3}),
        ToolCall(id="search:1", name="search", args={"q": "x"}),
        Wait(),
    ],
)
for violation in check(bad_args).errors:
    print(f"{violation.rule:30} {violation.message}")

tool-call/args-invalid         $.city: 3 is not of type 'string'
tool-call/undeclared-name      tool 'search' is not declared in the system prompt


**Tool-result echo discipline**: a result closes exactly one
pending call, byte-exactly; a second result for the same call is
an error, and a call left open at the end of the transcript is
a warning. Passing `checkers=[...]` runs a single rule family
instead of the default set:

In [7]:
echoes = Conversation(
    items=[
        ToolCall(id="bash:0", name="bash", args={}),
        Wait(),
        ToolResult(id="bash:0", payload="ok"),
        ToolResult(id="bash:0", payload="again"),  # duplicate: error
        ToolResult(id="bash:9", payload="orphan"),  # unknown id: error
        ToolCall(id="bash:1", name="bash", args={}),  # stays open: warning
        Wait(),
    ]
)
report = check(echoes, checkers=[ToolResultChecker()])
print("errors:")
for violation in report.errors:
    print(f"  {violation.rule:25} at {violation.location}")
print("warnings:")
for violation in report.warnings:
    print(f"  {violation.rule:25} at {violation.location}")

errors:
  tool-result/duplicate     at items[3]
  tool-result/unknown-id    at items[4]
warnings:
  tool-result/open-call     at items[5]


**Id convention** (warnings while the echo shape is unsettled
upstream): the model conventionally writes
`NAME:COUNTER` ids with one global counter. An OpenAI-style id is
accepted by the model but flagged as non-canonical:

In [8]:
openai_style = Conversation(
    items=[
        ToolCall(id="call_abc123", name="bash", args={}),
        Wait(),
    ]
)
report = check(openai_style, checkers=[IdConventionChecker()])
for violation in report.warnings:
    print(f"{violation.severity}: {violation.rule}: {violation.message}")
print("still conformant (warnings never break):", report.ok)

still conformant (warnings never break): True


**Verifiable answer placement** (advisory warning): the
committed answer canonically precedes the assistant message of its
burst:

In [9]:
answer_late = Conversation(
    items=[
        User(payload=[TextPart(text="2+2?")]),
        Assistant(payload="It is 4."),
        VerifiableAnswer(payload="4"),
        Wait(),
    ]
)
for violation in check(answer_late).warnings:
    print(f"{violation.rule}: {violation.message}")

verifiable-answer/after-assistant: verifiable_answer follows an assistant message in the same burst


### Configuring severities

Defaults are strict; loosen a rule to non-breaking per call, or
configure a checker instance once and reuse it:

In [10]:
report = check(messy, overrides={"input-order/canonical-order": Severity.WARNING})
print("loosened:", report.ok, "with", len(report.warnings), "warning(s)")

lenient_order = InputOrderChecker(severity=Severity.WARNING)
print("whole-checker:", check(messy, checkers=[lenient_order]).ok)
targeted_order = InputOrderChecker(
    overrides={"input-order/canonical-order": Severity.WARNING}
)
print("per-rule:", check(messy, checkers=[targeted_order]).ok)

loosened: True with 1 warning(s)
whole-checker: True
per-rule: True


### The full rule table

Every rule is declared as data on its checker class; `rule_table()`
aggregates the declarations of all builtin checkers into a
`tuple[Rule, ...]`. Each `Rule` carries: `id` (the stable
contract), `description` (one line), and `default_severity` (the
strict baseline, before any overrides). Printing the full table:

In [11]:
checker_names = ", ".join(type(checker).__name__ for checker in default_checkers())
print("default checker classes:", checker_names)
columns = f"{'rule id':34} {'severity':9} description"
print(columns)
print("-" * len(columns))
for rule in rule_table():
    print(f"{rule.id:34} {rule.default_severity:9} {rule.description}")
print(f"\n{len(rule_table())} rules")

default checker classes: SystemPromptChecker, InputOrderChecker, ToolCallChecker, ToolResultChecker, IdConventionChecker, WaitChecker, VerifiableAnswerChecker
rule id                            severity  description
--------------------------------------------------------
system/duplicate-tool-name         error     each declared tool name appears exactly once
input-order/canonical-order        error     inputs at one boundary appear in canonical delivery order
tool-call/undeclared-name          error     a tool call names a declared tool
tool-call/args-invalid             error     tool call args conform to the declared schema
tool-call/broken-schema            error     every declared tool schema is a valid JSON Schema
tool-result/unknown-id             error     a result echoes the id of a pending call
tool-result/duplicate              error     exactly one result closes a call
tool-result/open-call              warning   a call is still open at the end of the transcript
tool-call-

Because the table is plain data, filtering is ordinary Python;
for example, only the advisory (non-breaking) rules:

In [12]:
for rule in rule_table():
    if rule.default_severity is Severity.WARNING:
        print(rule.id)

tool-result/open-call
tool-call-id/non-canonical
tool-call-id/name-mismatch
tool-call-id/duplicate-counter
tool-call-id/non-sequential
wait/unterminated-transcript
verifiable-answer/after-assistant
verifiable-answer/multiple


## 3. Both levels together

A conversation can be perfectly valid message by message and still
non-conformant as a whole. Message-level validation cannot see
cross-message context; `check()` can:

In [13]:
# every message below is individually valid...
conversation = Conversation(
    system=system,
    items=[
        User(payload=[TextPart(text="weather in Bern?")]),
        Harness(payload="notice"),  # wrong side of the user message
        ToolCall(id="get_weather:0", name="get_weather", args={}),  # missing city
        Wait(),
        ToolResult(id="get_weather:0", payload='{"tempC": 21}'),
        Assistant(payload="21 degrees."),
        Wait(),
    ],
)
for violation in check(conversation):
    print(f"{violation.severity.upper():7} {violation.rule:32} at {violation.location}")

ERROR   input-order/canonical-order      at items[1]
ERROR   tool-call/args-invalid           at items[2].args


Fix both findings and the report comes back clean:

In [14]:
fixed = Conversation(
    system=system,
    items=[
        Harness(payload="notice"),
        User(payload=[TextPart(text="weather in Bern?")]),
        ToolCall(id="get_weather:0", name="get_weather", args={"city": "Bern"}),
        Wait(),
        ToolResult(id="get_weather:0", payload='{"tempC": 21}'),
        Assistant(payload="21 degrees."),
        Wait(),
    ],
)
report = check(fixed)
print("conformant:", report.ok, "| violations:", len(report))

conformant: True | violations: 0


## 4. Extending and serializing reports

Subclass `Checker`, declare stable `Rule` entries, and yield findings with `self.violation(...)`. An explicit `checkers=` sequence replaces the defaults, so extend them with `[*default_checkers(), custom]`. Reports are frozen dataclasses; use `dataclasses.asdict()` before JSON serialization.

In [15]:
class TranscriptSizeChecker(Checker):
    rules: ClassVar[tuple[Rule, ...]] = (
        Rule(
            "example/transcript-size", "the transcript stays compact", Severity.WARNING
        ),
    )

    def check(self, conversation: Conversation) -> Iterable[Violation]:
        if len(conversation.items) > 5:
            yield self.violation(
                "example/transcript-size",
                Location(),
                f"transcript has {len(conversation.items)} items",
            )


extended: ConformanceReport = check(
    fixed,
    checkers=[*default_checkers(), TranscriptSizeChecker()],
)
print(json.dumps(asdict(extended), indent=2))

{
  "violations": [
    {
      "rule": "example/transcript-size",
      "severity": "warning",
      "location": {
        "scope": "items",
        "item": null,
        "field": null
      },
      "message": "transcript has 7 items"
    }
  ]
}


That is the whole checker surface: construction raises on invalid
fields, `check()` reports on invalid relationships, severities bend
per checker, per rule, or per call, and new rule families are one
small subclass away. `make rules` prints the rule table from the
command line; the builtin implementations under
`apertus_common.checker.rules` are one worked example per rule
family.